[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/28_neural_networks/01_activation_functions.ipynb)

# Bài 01 — Hàm kích hoạt (Activation Functions)

> Dựa trên bài *Activation Function* (Tuần 5) của khoá AIVN-Machine-Learning, bổ sung: đạo hàm, vấn đề bão hoà và **vanishing gradient**, **dying ReLU**, các hàm hiện đại (GELU, SiLU), và thí nghiệm huấn luyện so sánh.

Trong bài này, chúng ta tìm hiểu các hàm kích hoạt phổ biến và cài đặt chúng với đầu vào là một mảng số (bài tập lập trình chi tiết ở `91_assignment_activation_functions`). Hàm kích hoạt đưa **tính phi tuyến** vào mạng — không có nó, nhiều lớp tuyến tính chồng lên nhau vẫn chỉ là **một** phép biến đổi tuyến tính (bài 01 chương 27).

<img src="images/activation-function-2.png" style="width:50%;height:50%;">

**🎯 Mục tiêu**
- Sigmoid, tanh, ReLU, PReLU/Leaky ReLU, ELU, Softplus (theo AIVN) + GELU, SiLU: công thức, đạo hàm, miền giá trị.
- Bão hoà và **vanishing gradient**; tâm 0; **dying ReLU**.
- Chọn hàm kích hoạt cho lớp ẩn và lớp ra.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/28_neural_networks

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler

from ml_course.visualization import set_style

try:
    import torch
    from torch import nn
except ImportError as e:
    raise ImportError("Chương này cần PyTorch: pip install -e \".[dl]\"") from e

set_style()
data = np.asarray([1, 5, -4, 3, -2])                                    # dữ liệu ví dụ của bài gốc

## 1. Hàm sigmoid (logistic)

<img src="images/sigmoid_function.png" style="width:50%;height:50%;">

$$\sigma(x) = \frac{1}{1 + e^{-x}}, \qquad \sigma'(x) = \sigma(x)\,(1 - \sigma(x))$$

**Phân tích hàm sigmoid:**
- $x \in \mathbb R$, $y \in (0, 1)$: ánh xạ $(-\infty, \infty) \to (0, 1)$ — diễn giải được như xác suất (lớp ra của phân loại nhị phân).
- Hàm đơn điệu tăng: nếu $x_1 \le x_2$ thì $f(x_1) \le f(x_2)$.
- Độ dốc (đạo hàm) lớn quanh $x = 0$ và **rất nhỏ ở hai đầu** (bão hoà): khi đầu vào lớn về trị tuyệt đối, gradient gần 0 → mạng học rất chậm. Đạo hàm lớn nhất chỉ là $0.25$.
- Đầu ra luôn dương (không có tâm 0) → gradient của các trọng số cùng một nơ-ron cùng dấu, đường đi tối ưu "zíc-zắc".

<img src="images/chap2_activation_sigmoid_plot.png" style="width:50%;height:50%;">

In [ ]:
def sigmoid_function(data):
    result = 1 / (1 + np.exp(-data))
    return result


print(sigmoid_function(data))

## 2. Hàm tanh

$$\tanh(x) = \frac{e^x - e^{-x}}{e^x + e^{-x}} = 2\sigma(2x) - 1, \qquad \tanh'(x) = 1 - \tanh^2(x)$$

Ví dụ tính `tanh` cho một danh sách:

<img src="images/activation_tanh.png" style="width:30%;height:30%;">

Đồ thị hàm `tanh` và đạo hàm:

<img src="images/chap2_activation_tanh_plot.png" style="width:50%;height:50%;">

- Miền giá trị $(-1, 1)$, **có tâm 0** → tốt hơn sigmoid cho lớp ẩn; đạo hàm lớn nhất bằng 1.
- Vẫn **bão hoà** ở hai đầu.

## 3. Hàm ReLU

$$\text{ReLU}(x) = \max(0, x), \qquad \text{ReLU}'(x) = \begin{cases}1 & x > 0\\0 & x < 0\end{cases}$$

<img src="images/activation_relu.png" style="width:30%;height:30%;">

<img src="images/chap2_activation_relu_plot.png" style="width:50%;height:50%;">

- Không bão hoà ở phía dương (gradient = 1) → huấn luyện mạng sâu nhanh hơn nhiều (Krizhevsky et al., 2012 — AlexNet); tính toán rẻ.
- Đầu ra **thưa** (nhiều số 0).
- Phía âm gradient bằng 0 → nơ-ron có thể "**chết**" (dying ReLU, mục 7).

## 4. Hàm PReLU / Leaky ReLU

$$\text{PReLU}(x) = \begin{cases}x & x > 0\\\alpha x & x \le 0\end{cases}$$

<img src="images/chap2_activation_prelu.png" style="width:30%;height:30%;">

Ví dụ với $\alpha = 0.1$:

<img src="images/activation_prelu.png" style="width:30%;height:30%;">

Leaky ReLU: $\alpha$ cố định nhỏ (0.01); **PReLU** (He et al., 2015): $\alpha$ là tham số **học được**. Gradient phía âm khác 0 → nơ-ron không chết hẳn.

## 5. Hàm ELU

$$\text{ELU}(x) = \begin{cases}x & x > 0\\\alpha(e^x - 1) & x \le 0\end{cases}$$

<img src="images/chap2_activation_elu.png" style="width:30%;height:30%;">

Ví dụ với $\alpha = 0.1$:

<img src="images/activation_elu.png" style="width:30%;height:30%;">

Phía âm **trơn** và bão hoà về $-\alpha$ → trung bình đầu ra gần 0 hơn ReLU (Clevert et al., 2015); tốn thêm phép `exp`.

## 6. Hàm Softplus

$$\text{softplus}(x) = \ln(1 + e^x), \qquad \text{softplus}'(x) = \sigma(x)$$

<img src="images/chap2_activation_softplus.png" style="width:30%;height:30%;">

<img src="images/activation_softplus.png" style="width:30%;height:30%;">

Phiên bản **trơn** của ReLU. Thường dùng khi cần đầu ra **dương** (vd độ lệch chuẩn trong mô hình xác suất) hơn là cho lớp ẩn.

### Các hàm hiện đại: GELU, SiLU

- **GELU** (Hendrycks & Gimpel, 2016): $x\,\Phi(x)$ ($\Phi$ = CDF chuẩn) — mặc định trong Transformer (BERT, GPT — chương 31).
- **SiLU / Swish** (Ramachandran et al., 2017): $x\,\sigma(x)$ — dùng trong EfficientNet, LLaMA (SwiGLU).
Cả hai giống ReLU khi $|x|$ lớn nhưng trơn, hơi âm quanh 0.

In [ ]:
xs = np.linspace(-5, 5, 500)
funcs = {
    "sigmoid": (lambda x: 1 / (1 + np.exp(-x))),
    "tanh": np.tanh,
    "ReLU": (lambda x: np.maximum(0, x)),
    "Leaky ReLU (0.1)": (lambda x: np.where(x > 0, x, 0.1 * x)),
    "ELU (1.0)": (lambda x: np.where(x > 0, x, np.exp(np.minimum(x, 0)) - 1)),
    "softplus": (lambda x: np.log1p(np.exp(x))),
    "GELU": (lambda x: torch.nn.functional.gelu(torch.tensor(x)).numpy()),
    "SiLU": (lambda x: x / (1 + np.exp(-x))),
}
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
for ax, (name, fn) in zip(axes.ravel(), funcs.items()):
    y = fn(xs)
    dy = np.gradient(y, xs)                                             # đạo hàm số
    ax.plot(xs, y, label="f(x)")
    ax.plot(xs, dy, "--", label="f'(x)")
    ax.axhline(0, color="gray", lw=0.5)
    ax.set_title(name)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()
for name in ["sigmoid", "tanh", "ReLU"]:
    print(f"{name:8s}: đạo hàm lớn nhất ≈ {np.max(np.gradient(funcs[name](xs), xs)):.3f} | f'(5) ≈ {np.gradient(funcs[name](xs), xs)[-1]:.4f}")

## 7. Vanishing gradient

Backprop nhân các đạo hàm qua từng lớp. Với sigmoid, mỗi lớp nhân với $\sigma'(z) \le 0.25$ → sau 10 lớp gradient có thể nhỏ đi hàng nghìn lần: các lớp **đầu** gần như không học.

Đo chuẩn gradient của từng lớp trong mạng 10 lớp ẩn (mỗi lớp 64 nơ-ron) trên một batch dữ liệu chữ số. Để so sánh **công bằng**, mỗi hàm kích hoạt dùng cách khởi tạo phù hợp với nó (He cho ReLU/GELU, Xavier cho sigmoid/tanh — lý do ở bài 02):

In [ ]:
X_dg, y_dg = load_digits(return_X_y=True)
X_t = torch.tensor(StandardScaler().fit_transform(X_dg), dtype=torch.float32)
y_t = torch.tensor(y_dg)


acts = {"Sigmoid": nn.Sigmoid, "Tanh": nn.Tanh, "ReLU": nn.ReLU, "GELU": nn.GELU}
INIT = {nn.Sigmoid: ("xavier", 1.0), nn.Tanh: ("xavier", 5 / 3), nn.ReLU: ("he", None), nn.GELU: ("he", None)}


def deep_mlp(act_cls, depth=10, width=64, seed=0):
    torch.manual_seed(seed)
    layers, d = [], 64
    for _ in range(depth):
        lin = nn.Linear(d, width)
        kind, gain = INIT[act_cls]
        if kind == "he":
            nn.init.kaiming_normal_(lin.weight, nonlinearity="relu")
        else:
            nn.init.xavier_normal_(lin.weight, gain=gain)
        nn.init.zeros_(lin.bias)
        layers += [lin, act_cls()]
        d = width
    layers.append(nn.Linear(d, 10))
    return nn.Sequential(*layers)
fig, ax = plt.subplots(figsize=(9, 4))
for name, act in acts.items():
    model = deep_mlp(act)
    loss = nn.functional.cross_entropy(model(X_t[:256]), y_t[:256])
    loss.backward()
    norms = [m.weight.grad.norm().item() for m in model if isinstance(m, nn.Linear)]
    ax.semilogy(range(1, len(norms) + 1), norms, marker="o", label=name)
    print(f"{name:8s}: ‖grad‖ lớp 1 = {norms[0]:.2e} | lớp cuối = {norms[-1]:.2e} | tỉ lệ = {norms[0] / norms[-1]:.1e}")
ax.set_xlabel("lớp (1 = gần đầu vào)")
ax.set_ylabel("‖∂L/∂W‖ (log)")
ax.legend()
plt.show()

Với sigmoid, gradient ở lớp đầu nhỏ hơn lớp cuối khoảng **một triệu lần** — lớp đầu gần như đứng yên, dù đã khởi tạo hợp lý. Tanh, ReLU, GELU giữ gradient cùng bậc độ lớn qua 10 lớp. (Khởi tạo, chuẩn hoá — bài 02 — và kết nối tắt — residual, chương 29 — là phần còn lại của lời giải cho mạng rất sâu.)

## 8. So sánh khi huấn luyện

Cùng mạng 6 lớp ẩn × 64, cùng SGD (lr = 0.05, 15 epoch), chỉ khác hàm kích hoạt:

In [ ]:
def train_curve(act_cls, depth=6, epochs=15, lr=0.05, seed=0):
    model = deep_mlp(act_cls, depth=depth, seed=seed)
    opt = torch.optim.SGD(model.parameters(), lr=lr)
    g = torch.Generator().manual_seed(seed)
    losses = []
    for _ in range(epochs):
        perm = torch.randperm(len(X_t), generator=g)
        for s in range(0, len(perm), 32):
            idx = perm[s:s + 32]
            opt.zero_grad()
            nn.functional.cross_entropy(model(X_t[idx]), y_t[idx]).backward()
            opt.step()
        with torch.no_grad():
            losses.append(nn.functional.cross_entropy(model(X_t), y_t).item())
    return losses


fig, ax = plt.subplots(figsize=(9, 4))
for name, act in acts.items():
    curve = train_curve(act)
    ax.plot(range(1, len(curve) + 1), curve, marker=".", label=name)
    print(f"{name:8s}: loss sau epoch 1 = {curve[0]:.3f} | sau epoch 15 = {curve[-1]:.4f}")
ax.set_yscale("log")
ax.set_xlabel("epoch")
ax.set_ylabel("loss train (log)")
ax.legend()
plt.show()

Mạng sigmoid 6 lớp **không học** được với cùng thiết lập (loss đứng yên ở $\ln 10 \approx 2.3$ — đoán mò 10 lớp), trong khi tanh, ReLU, GELU hội tụ nhanh; ReLU và GELU đạt loss thấp nhất.

## 9. Dying ReLU

Nếu một nơ-ron ReLU rơi vào trạng thái $z < 0$ với **mọi** đầu vào (vd sau một bước cập nhật quá lớn đẩy bias rất âm), gradient của nó luôn bằng 0 → không bao giờ hồi phục. Mạng 3 lớp ẩn × 256 ReLU, huấn luyện 300 bước bằng Adam với learning rate tăng dần; đếm tỉ lệ nơ-ron không bao giờ dương trên toàn bộ dữ liệu ở từng lớp:

In [ ]:
def dead_fractions(lr, width=256, depth=3, steps=300, seed=0):
    torch.manual_seed(seed)
    layers, d = [], 64
    for _ in range(depth):
        lin = nn.Linear(d, width)
        nn.init.kaiming_normal_(lin.weight, nonlinearity="relu")
        nn.init.zeros_(lin.bias)
        layers += [lin, nn.ReLU()]
        d = width
    model = nn.Sequential(*layers, nn.Linear(d, 10))
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for _ in range(steps):
        idx = torch.randint(0, len(X_t), (64,))
        opt.zero_grad()
        nn.functional.cross_entropy(model(X_t[idx]), y_t[idx]).backward()
        opt.step()
    fracs, h = [], X_t
    with torch.no_grad():
        for layer in model[:-1]:
            if isinstance(layer, nn.Linear):
                pre = layer(h)
                fracs.append((pre <= 0).all(dim=0).float().mean().item())   # không bao giờ dương → gradient luôn 0
                h = torch.relu(pre)
        final_loss = nn.functional.cross_entropy(model(X_t), y_t).item()
    return fracs, final_loss


for lr in [0.001, 0.01, 0.05]:
    fracs, final_loss = dead_fractions(lr)
    print(f"Adam lr = {lr:<5}: nơ-ron chết ở lớp 1/2/3 = {[f'{f:.0%}' for f in fracs]} | loss train cuối = {final_loss:.3f}")

Learning rate càng lớn, càng nhiều nơ-ron ở các lớp sâu chết vĩnh viễn — với lr = 0.05 phần lớn lớp 2 đã chết và mạng mất khả năng học (loss cao). Leaky ReLU/ELU/GELU vẫn còn gradient ở phía âm nên nơ-ron có thể thoát ra. Phòng tránh: learning rate hợp lý (bài 03), khởi tạo đúng (bài 02), batch norm, hoặc dùng Leaky ReLU/ELU/GELU.

## 10. Chọn hàm kích hoạt

| Vị trí | Lựa chọn |
|---|---|
| Lớp ẩn — mặc định | **ReLU** (đơn giản, nhanh); GELU/SiLU trong Transformer, mạng hiện đại |
| Lớp ẩn — lo dying ReLU | Leaky ReLU, PReLU, ELU |
| Lớp ẩn — RNN (cổng) | tanh, sigmoid (cổng LSTM/GRU — chương 30) |
| Lớp ra — hồi quy | không kích hoạt (tuyến tính); softplus nếu cần dương |
| Lớp ra — nhị phân | sigmoid (thường gộp vào `BCEWithLogitsLoss`) |
| Lớp ra — đa lớp | softmax (gộp vào `CrossEntropyLoss`) |

Tránh sigmoid/tanh cho lớp ẩn của mạng sâu feed-forward.

## 🧠 Kiểm tra nhanh

1. Vì sao mạng không có hàm kích hoạt phi tuyến chỉ tương đương một lớp tuyến tính?
2. Đạo hàm lớn nhất của sigmoid là bao nhiêu? Hệ quả với mạng sâu?
3. Tanh hơn sigmoid ở điểm nào cho lớp ẩn?
4. Dying ReLU là gì? Cách phòng tránh?
5. Hàm kích hoạt nào cho lớp ra của bài toán phân loại 10 lớp? Hồi quy?

## 📌 Tóm tắt

- Sigmoid/tanh bão hoà → vanishing gradient; tanh có tâm 0.
- ReLU không bão hoà phía dương, nhanh, nhưng có thể chết; Leaky/PReLU/ELU sửa phía âm; GELU/SiLU trơn, dùng trong mạng hiện đại.
- Lớp ra chọn theo bài toán (tuyến tính, sigmoid, softmax).

## 📚 Tài liệu tham khảo

- AIVN-Machine-Learning — Week 5: *Activation Function* (github.com/thanhhff/AIVN-Machine-Learning)
- Glorot, Bordes & Bengio (2011) — *Deep Sparse Rectifier Neural Networks*
- He et al. (2015) — *Delving Deep into Rectifiers* (PReLU)
- Clevert, Unterthiner & Hochreiter (2015) — *Fast and Accurate Deep Network Learning by ELUs*
- Hendrycks & Gimpel (2016) — *Gaussian Error Linear Units (GELUs)*